# Stellar-Qwen3-8B-ARM — RewardBench 1 Evaluation

- **Model:** [`Stellar-lab/Stellar-Qwen3-8B-ARM`](https://huggingface.co/Stellar-lab/Stellar-Qwen3-8B-ARM)
- **Dataset:** [allenai/reward-bench](https://huggingface.co/datasets/allenai/reward-bench) (2,985 filtered pairs across 23 subsets)
- **Reported Result:** **89.08%** Subset Macro-Average (`85.52%` with 50/50 Math+Code Reasoning weighting)
- **Hardware:** Requires a CUDA GPU (`>=2` GPUs run FP16 via `device_map="auto"`; `1` GPU runs 4-bit NF4 quantization)

In [1]:
# ==============================================================================
# CELL 0: Dependencies Installation & Environment Setup
# ==============================================================================
!pip install -q --upgrade \
    "bitsandbytes>=0.46.1" \
    "accelerate>=0.33.0" \
    "transformers>=4.44.0" \
    "datasets>=2.20.0" \
    "tiktoken" \
    "sentencepiece" \
    "seaborn" \
    "scipy" \
    "tabulate"

import sys
import torch
import transformers
import bitsandbytes as bnb
import accelerate

print("=" * 80)
print("REWARDBENCH 1 EVALUATION ENVIRONMENT")
print("=" * 80)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers:         {transformers.__version__}")
print(f"BitsAndBytes:         {bnb.__version__}")
print(f"Accelerate:           {accelerate.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()} ({torch.cuda.device_count()} GPU(s))")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GiB")
print("=" * 80)

REWARDBENCH 1 EVALUATION ENVIRONMENT
Python Version:       3.13.16
PyTorch Version:      2.11.0+cu130 (CUDA: 13.0)
Transformers:         5.22.2
BitsAndBytes:         0.50.2
Accelerate:           1.15.0
CUDA Available:       True (2 GPU(s))
  GPU 0: Tesla T4 | VRAM: 14.56 GiB
  GPU 1: Tesla T4 | VRAM: 14.56 GiB


In [2]:
# ==============================================================================
# CELL 1: Load Stellar-Qwen3-8B-ARM from Hugging Face Hub
# ==============================================================================
import os
import sys
import gc
import math
import types
import torch
import transformers.modeling_utils
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "Stellar-lab/Stellar-Qwen3-8B-ARM"
print(f"[MODEL] Loading model from Hugging Face Hub: {MODEL_ID}")

class TiedWeightsDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, "_all_tied_weights_keys_storage", {})
    def __set__(self, instance, value):
        instance._all_tied_weights_keys_storage = value

transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = TiedWeightsDescriptor()
transformers.modeling_utils.PreTrainedModel._tied_weights_keys = {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"
print(f"[TOKENIZER] Vocab size: {len(tokenizer):,} | Pad Token ID: {tokenizer.pad_token_id}")

num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    print(f"[HARDWARE] Loading FP16 model partitioned across {num_gpus} GPUs...")
    max_memory = {i: "13.0GiB" for i in range(num_gpus)}
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
    )
elif num_gpus == 1:
    print("[HARDWARE] Single GPU detected -> Loading in 4-bit NF4...")
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_cfg,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
else:
    raise RuntimeError("A CUDA GPU is required for evaluation.")

def multi_gpu_forward_dag_head_only(self, last_6_hidden: torch.Tensor):
    head_dev = self.dag_down_proj.weight.device
    head_dtype = self.dag_down_proj.weight.dtype
    h_in = last_6_hidden.to(device=head_dev, dtype=head_dtype)

    z = self.dag_norm(self.dag_down_proj(h_in))
    for layer in self.dag_layers:
        z = layer(z, associative=self.config.associative)

    raw_logits = []
    for i in range(self.config.num_dimensions):
        head_i = self.dim_heads[i]
        h_dim = z[:, i, :].to(device=head_i.weight.device, dtype=head_i.weight.dtype)
        raw_logits.append(head_i(h_dim).to(device=head_dev))

    raw_z = torch.cat(raw_logits, dim=-1).float()
    z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)
    agg_weights = torch.nn.functional.softmax(self.agg_logits.to(device=head_dev).float(), dim=0)

    if self.config.scalar_only_mode:
        dim_logits = z_clamped
        dim_scores = torch.sigmoid(dim_logits)
        scalar_logit = (dim_logits * agg_weights).sum(dim=-1)
        return {
            "scalar_logit": scalar_logit,
            "dim_logits": dim_logits,
            "dim_scores": dim_scores,
            "agg_weights": agg_weights,
            "veto_penalty": torch.zeros_like(scalar_logit),
        }

    if self.config.associative:
        z3 = z_clamped[:, 2:3]
        z4 = z_clamped[:, 3:4]
        tau_m = self.config.monotonic_tau
        z_prereq = (
            -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True)
            + tau_m * math.log(2.0)
        )
        z5_mono = torch.clamp(
            z_prereq + self.config.monotonic_delta_tol - torch.nn.functional.softplus(-raw_z[:, 4:5]),
            min=-15.0,
            max=15.0,
        )
        dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
    else:
        dim_logits = z_clamped

    dim_scores = torch.sigmoid(dim_logits)
    linear_term = self.config.beta_lin * (dim_logits * agg_weights).sum(dim=-1)

    gk_logits = dim_logits[:, self.config.gatekeeper_indices]
    kappa = self.config.veto_kappa
    veto_lambda = torch.nn.functional.softplus(self.veto_raw_lambda.to(device=head_dev).float())
    veto_thresh = self.veto_thresholds.to(device=head_dev).float().unsqueeze(0)

    exponents = -kappa * (gk_logits - veto_thresh)
    zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=head_dev)
    veto_penalty = (veto_lambda / kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)

    scalar_logit = linear_term - veto_penalty
    return {
        "scalar_logit": scalar_logit,
        "dim_logits": dim_logits,
        "dim_scores": dim_scores,
        "agg_weights": agg_weights,
        "veto_penalty": veto_penalty,
    }

model.forward_dag_head_only = types.MethodType(multi_gpu_forward_dag_head_only, model)
model.eval()
input_device = model.backbone.get_input_embeddings().weight.device

print("=" * 80)
print(f"[MODEL READY] Loaded {model.__class__.__name__}")
for i in range(num_gpus):
    vram_gb = torch.cuda.memory_allocated(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"  GPU {i} Allocated VRAM: {vram_gb:.2f} GB / {total_gb:.2f} GB")
print("=" * 80)

[MODEL READY] Loaded StellarQwen3ARMForRewardModeling
  GPU 0 Allocated VRAM: 7.99 GB / 14.56 GB
  GPU 1 Allocated VRAM: 6.14 GB / 14.56 GB


In [3]:
# ==============================================================================
# CELL 2: Load Official `allenai/reward-bench` Dataset (2,985 Filtered Pairs)
# ==============================================================================
import os
import numpy as np
import pandas as pd
from datasets import load_dataset

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PREDICTIONS_PATH = os.path.join(OUTPUT_DIR, "rewardbench1_predictions.parquet")
LEADERBOARD_CSV_PATH = os.path.join(OUTPUT_DIR, "rewardbench1_results.csv")
LEADERBOARD_PNG_PATH = os.path.join(OUTPUT_DIR, "rewardbench1_comparison.png")

print("[DATASET] Loading `allenai/reward-bench` (filtered split)...")
try:
    rb_dataset = load_dataset("allenai/reward-bench", split="filtered")
except Exception:
    rb_dataset = load_dataset("allenai/reward-bench")["eval"]
df_rb = rb_dataset.to_pandas()

print("=" * 80)
print(f"[REWARDBENCH 1 LOADED] Total evaluation pairs: {len(df_rb):,} across {df_rb['subset'].nunique()} subsets")
print("-" * 80)
for subset, count in df_rb["subset"].value_counts().items():
    print(f"  - {subset:<32s}: {count:>4d} pairs")
print("=" * 80)

[DATASET] Loading `allenai/reward-bench` (filtered split)...


[REWARDBENCH 1 LOADED] Total evaluation pairs: 2,985 across 23 subsets
--------------------------------------------------------------------------------
  - math-prm                        :  447 pairs
  - xstest-should-respond           :  250 pairs
  - hep-python                      :  164 pairs
  - hep-java                        :  164 pairs
  - hep-js                          :  164 pairs
  - hep-cpp                         :  164 pairs
  - hep-go                          :  164 pairs
  - hep-rust                        :  164 pairs
  - xstest-should-refuse            :  154 pairs
  - donotanswer                     :  136 pairs
  - llmbar-adver-neighbor           :  134 pairs
  - refusals-dangerous              :  100 pairs
  - llmbar-natural                  :  100 pairs
  - alpacaeval-easy                 :  100 pairs
  - refusals-offensive              :  100 pairs
  - alpacaeval-hard                 :   95 pairs
  - alpacaeval-length               :   95 pairs
  - llmbar-adve

In [4]:
# ==============================================================================
# CELL 3: Batched Inference Across All 2,985 RewardBench 1 Pairs
# ==============================================================================
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Tuple

BATCH_SIZE = 2
MAX_LEN = 1280
RESP_RESERVE = 896

def encode_tail_safe(prompt: str, response: str) -> List[int]:
    p_clean = str(prompt).strip()
    if not p_clean.startswith("User:"):
        p_clean = f"User: {p_clean}\nAssistant: "
    p_ids = tokenizer.encode(p_clean, add_special_tokens=False)
    r_ids = tokenizer.encode(str(response).strip(), add_special_tokens=False)

    if len(p_ids) + len(r_ids) <= MAX_LEN:
        return p_ids + r_ids

    r_keep = r_ids[-min(len(r_ids), RESP_RESERVE):]
    p_budget = MAX_LEN - len(r_keep)
    p_keep = p_ids[-p_budget:] if p_budget > 0 else []
    return p_keep + r_keep

def left_pad_sequences(seq_lists: List[List[int]], pad_id: int, dev: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    max_l = max(len(s) for s in seq_lists)
    b_sz = len(seq_lists)
    padded = torch.full((b_sz, max_l), pad_id, dtype=torch.long, device=dev)
    mask = torch.zeros((b_sz, max_l), dtype=torch.long, device=dev)
    for i, ids in enumerate(seq_lists):
        pad_len = max_l - len(ids)
        padded[i, pad_len:] = torch.tensor(ids, dtype=torch.long, device=dev)
        mask[i, pad_len:] = 1
    return padded, mask

print(f"[EVALUATION] Running RewardBench 1 inference (Batch Size: {BATCH_SIZE}, Max Len: {MAX_LEN})...")
results = []
rows = df_rb.to_dict("records")
pad_token_id = tokenizer.pad_token_id

for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="RewardBench 1"):
    batch_rows = rows[i : i + BATCH_SIZE]

    c_lists = [encode_tail_safe(r["prompt"], r["chosen"]) for r in batch_rows]
    r_lists = [encode_tail_safe(r["prompt"], r["rejected"]) for r in batch_rows]

    c_ids, c_mask = left_pad_sequences(c_lists, pad_token_id, input_device)
    r_ids, r_mask = left_pad_sequences(r_lists, pad_token_id, input_device)

    with torch.no_grad():
        out_c = model(input_ids=c_ids, attention_mask=c_mask)
        out_r = model(input_ids=r_ids, attention_mask=r_mask)

    logits_c = out_c.logits.float().cpu().numpy()
    logits_r = out_r.logits.float().cpu().numpy()
    veto_c = out_c.veto_penalty.float().cpu().numpy()
    veto_r = out_r.veto_penalty.float().cpu().numpy()
    scores_c = out_c.dim_scores.float().cpu().numpy()
    scores_r = out_r.dim_scores.float().cpu().numpy()

    for idx, r in enumerate(batch_rows):
        sc_c = float(logits_c[idx])
        sc_r = float(logits_r[idx])
        is_correct = 1.0 if sc_c > sc_r else (0.5 if abs(sc_c - sc_r) <= 1e-6 else 0.0)

        results.append({
            "id": r.get("id", i + idx),
            "subset": str(r["subset"]),
            "score_chosen": round(sc_c, 4),
            "score_rejected": round(sc_r, 4),
            "margin": round(sc_c - sc_r, 4),
            "is_correct": float(is_correct),
            "veto_chosen": round(float(veto_c[idx]), 4),
            "veto_rejected": round(float(veto_r[idx]), 4),
            "d1_adh_chosen": round(float(scores_c[idx, 0]), 4),
            "d1_adh_rejected": round(float(scores_r[idx, 0]), 4),
            "d4_reason_chosen": round(float(scores_c[idx, 3]), 4),
            "d4_reason_rejected": round(float(scores_r[idx, 3]), 4),
            "d5_corr_chosen": round(float(scores_c[idx, 4]), 4),
            "d5_corr_rejected": round(float(scores_r[idx, 4]), 4),
        })

df_eval = pd.DataFrame(results)
df_eval.to_parquet(PREDICTIONS_PATH, index=False)
print(f"[SAVED] Processed {len(df_eval):,} pairs -> `{PREDICTIONS_PATH}`.")

[SAVED] Processed 2,985 pairs -> `/kaggle/working/rewardbench1_predictions.parquet`.


In [5]:
# ==============================================================================
# CELL 4: Category Aggregation & Reference Comparison Table
# ==============================================================================
import numpy as np
import pandas as pd

df_eval = pd.read_parquet(PREDICTIONS_PATH)

REWARDBENCH_CATEGORIES = {
    "Chat": [
        "alpacaeval-easy", "alpacaeval-length", "alpacaeval-hard",
        "mt-bench-easy", "mt-bench-med",
    ],
    "Chat Hard": [
        "mt-bench-hard", "llmbar-natural", "llmbar-adver-neighbor",
        "llmbar-adver-GPTInst", "llmbar-adver-GPTOut", "llmbar-adver-manual",
    ],
    "Safety": [
        "refusals-dangerous", "refusals-offensive", "xstest-should-refuse",
        "xstest-should-respond", "donotanswer",
    ],
    "Reasoning": [
        "math-prm", "hep-cpp", "hep-go", "hep-java", "hep-js", "hep-python", "hep-rust",
    ],
}

subset_accs = df_eval.groupby("subset")["is_correct"].mean() * 100.0

chat_score = float(subset_accs[REWARDBENCH_CATEGORIES["Chat"]].mean())
chat_hard_score = float(subset_accs[REWARDBENCH_CATEGORIES["Chat Hard"]].mean())
safety_score = float(subset_accs[REWARDBENCH_CATEGORIES["Safety"]].mean())

hep_subsets = ["hep-cpp", "hep-go", "hep-java", "hep-js", "hep-python", "hep-rust"]
hep_code_mean = float(subset_accs[hep_subsets].mean())
math_prm_acc = float(subset_accs["math-prm"])

reasoning_macro = float(subset_accs[REWARDBENCH_CATEGORIES["Reasoning"]].mean())
reasoning_50_50 = 0.5 * math_prm_acc + 0.5 * hep_code_mean

overall_macro = float(np.mean([chat_score, chat_hard_score, safety_score, reasoning_macro]))
overall_50_50 = float(np.mean([chat_score, chat_hard_score, safety_score, reasoning_50_50]))

print("=" * 88)
print("REWARDBENCH 1 EVALUATION SUMMARY: Stellar-Qwen3-8B-ARM")
print("=" * 88)
print(f"  Chat                          : {chat_score:>6.2f}%")
print(f"  Chat Hard                     : {chat_hard_score:>6.2f}%")
print(f"  Safety                        : {safety_score:>6.2f}%")
print(f"  Reasoning (Code hep-* Mean)   : {hep_code_mean:>6.2f}%")
print(f"  Reasoning (math-prm)          : {math_prm_acc:>6.2f}%")
print(f"  Reasoning (7-Subset Macro)    : {reasoning_macro:>6.2f}%")
print(f"  Reasoning (50/50 Math+Code)   : {reasoning_50_50:>6.2f}%")
print("-" * 88)
print(f"  OVERALL (Subset Macro-Avg)    : {overall_macro:>6.2f}%")
print(f"  OVERALL (50/50 Math+Code Avg) : {overall_50_50:>6.2f}%")
print("=" * 88)

reference_models = [
    {"Model": "infly/INF-ORM-Llama3.1-70B", "Type": "70B Classifier", "Overall": 95.11, "Chat": 96.6, "Chat Hard": 91.0, "Safety": 93.6, "Reasoning": 99.1},
    {"Model": "Skywork/Skywork-Reward-Gemma-2-27B-v0.2", "Type": "27B Classifier", "Overall": 94.26, "Chat": 96.1, "Chat Hard": 89.9, "Safety": 93.0, "Reasoning": 98.1},
    {"Model": "Skywork/Skywork-Reward-Llama-3.1-8B-v0.2", "Type": "8B Classifier", "Overall": 93.13, "Chat": 94.7, "Chat Hard": 88.4, "Safety": 92.7, "Reasoning": 96.7},
    {"Model": "LxzGordon/URM-LLaMa-3.1-8B", "Type": "8B Classifier", "Overall": 92.94, "Chat": 95.5, "Chat Hard": 88.2, "Safety": 91.1, "Reasoning": 97.0},
    {"Model": "Ray2333/GRM-Llama3-8B-rewardmodel-ft", "Type": "8B Classifier", "Overall": 91.54, "Chat": 95.5, "Chat Hard": 86.2, "Safety": 90.8, "Reasoning": 93.6},
    {"Model": "RLHFlow/ArmoRM-Llama3-8B-v0.1", "Type": "8B MoE RM", "Overall": 90.40, "Chat": 96.9, "Chat Hard": 76.8, "Safety": 90.5, "Reasoning": 97.3},
    {"Model": "Stellar-Qwen3-8B-ARM (Macro Avg)", "Type": "8B Causal DAG (7.5k pairs)", "Overall": round(overall_macro, 2), "Chat": round(chat_score, 1), "Chat Hard": round(chat_hard_score, 1), "Safety": round(safety_score, 1), "Reasoning": round(reasoning_macro, 1)},
    {"Model": "Skywork/Skywork-Reward-Llama-3.1-8B", "Type": "8B Classifier", "Overall": 87.70, "Chat": 96.1, "Chat Hard": 80.4, "Safety": 87.8, "Reasoning": 86.3},
    {"Model": "openai/gpt-4o-2024-08-06", "Type": "Proprietary Judge", "Overall": 86.73, "Chat": 96.1, "Chat Hard": 76.1, "Safety": 88.1, "Reasoning": 86.6},
    {"Model": "Stellar-Qwen3-8B-ARM (50/50 Weighted)", "Type": "8B Causal DAG (7.5k pairs)", "Overall": round(overall_50_50, 2), "Chat": round(chat_score, 1), "Chat Hard": round(chat_hard_score, 1), "Safety": round(safety_score, 1), "Reasoning": round(reasoning_50_50, 1)},
]

df_leaderboard = pd.DataFrame(reference_models).sort_values(by="Overall", ascending=False).reset_index(drop=True)
df_leaderboard.to_csv(LEADERBOARD_CSV_PATH, index=False)

print("\nREWARDBENCH 1 REFERENCE COMPARISON")
print("=" * 105)
print(df_leaderboard.to_markdown(index=False))
print("=" * 105)

REWARDBENCH 1 EVALUATION SUMMARY: Stellar-Qwen3-8B-ARM
  Chat                          :  92.36%
  Chat Hard                     :  83.20%
  Safety                        :  90.21%
  Reasoning (Code hep-* Mean)   :  96.24%
  Reasoning (math-prm)          :  56.38%
  Reasoning (7-Subset Macro)    :  90.54%
  Reasoning (50/50 Math+Code)   :  76.31%
----------------------------------------------------------------------------------------
  OVERALL (Subset Macro-Avg)    :  89.08%
  OVERALL (50/50 Math+Code Avg) :  85.52%

REWARDBENCH 1 REFERENCE COMPARISON
| Model                                    | Type                       |   Overall |   Chat |   Chat Hard |   Safety |   Reasoning |
|:-----------------------------------------|:---------------------------|----------:|-------:|------------:|---------:|------------:|
| infly/INF-ORM-Llama3.1-70B               | 70B Classifier             |     95.11 |   96.6 |        91   |     93.6 |        99.1 |
| Skywork/Skywork-Reward-Gemma-2-27B-v0.

In [6]:
# ==============================================================================
# CELL 5: Export RewardBench 1 Category & Subset Breakdown Plot
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7.0), dpi=300, gridspec_kw={"width_ratios": [1.25, 1.0]})

df_plot = df_leaderboard.sort_values(by="Overall", ascending=True)
models = [m.split("/")[-1] for m in df_plot["Model"]]
overalls = df_plot["Overall"].tolist()
colors = ["#d95f02" if "Stellar" in m else ("#4a6fe3" if "8B" in m else "#8c8c8c") for m in df_plot["Model"]]

bars = ax1.barh(models, overalls, color=colors, height=0.65, edgecolor="black", linewidth=0.7)
ax1.set_xlim(70, 100)
ax1.set_xlabel("Overall Score (%)", fontsize=11, fontweight="bold")
ax1.set_title("A. RewardBench 1 Reference Comparison", fontsize=11, fontweight="bold")
ax1.grid(axis="x", linestyle="--", alpha=0.4)

for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.4, bar.get_y() + bar.get_height() / 2, f"{w:.2f}%", va="center", ha="left", fontsize=8.5, fontweight="bold")

cats = ["Chat", "Chat Hard", "Safety", "Reasoning\n(Macro)", "Reasoning\n(50/50)"]
vals = [chat_score, chat_hard_score, safety_score, reasoning_macro, reasoning_50_50]
bars2 = ax2.bar(cats, vals, color=["#4a6fe3", "#d95f02", "#5ab4ac", "#7570b3", "#e7298a"], width=0.55, edgecolor="black", linewidth=0.7)
ax2.set_ylim(50, 105)
ax2.set_ylabel("Accuracy (%)", fontsize=11, fontweight="bold")
ax2.set_title("B. Stellar-Qwen3-8B-ARM Category Breakdown", fontsize=11, fontweight="bold")
ax2.grid(axis="y", linestyle="--", alpha=0.4)

for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + 1.0, f"{h:.2f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
fig.savefig(LEADERBOARD_PNG_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Image(filename=LEADERBOARD_PNG_PATH))

### Comparison Figure

![Stellar-Qwen3-8B-ARM — RewardBench 1 Evaluation](assets/rewardbench1_comparison.png)